# Test GOES16 coordinates' transformation to pixel coordinates 

In [18]:
import numpy as np
from pyproj import Proj
import xarray as xr
import plotly.graph_objects as go
import cartopy.crs as ccrs

In [3]:
from src.co2sat.utils import project_root

In [4]:
path = (
    project_root()
    / "data"
    / "raw"
    / "goes16_test"
    / "OR_ABI-L1b-RadC-M6C01_G16_s20210911201147_e20210911203520_c20210911203556.nc"
)

In [5]:
def make_goes_projection(ds: xr.Dataset) -> Proj:
    """Build a pyproj projection object from GOES-16 metadata."""
    proj_info = ds["goes_imager_projection"]
    return Proj(
        proj="geos",
        h=proj_info.attrs["perspective_point_height"],
        lon_0=proj_info.attrs["longitude_of_projection_origin"],
        sweep=proj_info.attrs["sweep_angle_axis"],
        a=6378137.0,  # equatorial radius
        b=6356752.31414,  # polar radius
    )

In [6]:
def lonlat_to_xy(lon: float, lat: float, proj: Proj) -> tuple[float, float]:
    """Convert lon/lat in degrees to GOES-16 fixed-grid x,y in radians."""
    x_m, y_m = proj(lon, lat)
    # pyproj returns meters; convert to radians by dividing by satellite height
    h = proj.crs.to_dict()["h"]
    return x_m / h, y_m / h

Let's test these functions on the most famous power plant in the US: The Hoover dam, which is located in (36.0162° N, -114.7372° W)

In [8]:
ds = xr.open_dataset(path)

In [9]:
proj = make_goes_projection(ds)

In [10]:
x, y = lonlat_to_xy(-114.7372, 36.0162, proj)

/home/karim/co2-satellite-replication/.venv/lib/python3.12/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


In [11]:
print(f"Hoover dam in GOES fixed-grid: x={x:.6f}, y={y:.6f} radians")

Hoover dam in GOES fixed-grid: x=-0.085831, y=0.097345 radians


In [12]:
# Find the nearest pixel in the dataset
x_idx = np.abs(ds["x"].values - x).argmin()
y_idx = np.abs(ds["y"].values - y).argmin()

In [13]:
print(f"Nearest pixel index: x_idx={x_idx}, y_idx={y_idx}")

Nearest pixel index: x_idx=554, y_idx=1103


In [14]:
print(f"Radiance at that pixel: {ds['Rad'].values[y_idx, x_idx]}")

Radiance at that pixel: 0.05075645446777344


In [17]:
# Extract data
rad = ds["Rad"].values
x = ds["x"].values
y = ds["y"].values

fig = go.Figure()

# Main radiance image
fig.add_trace(
    go.Heatmap(
        z=rad,
        x=x,
        y=y,
        colorscale="Viridis",
        colorbar=dict(title="Radiance"),
    )
)

# Highlight point (Hoover Dam)
fig.add_trace(
    go.Scatter(
        x=[x[x_idx]],
        y=[y[y_idx]],
        mode="markers",
        marker=dict(color="red", size=12, symbol="x"),
        name="Hoover Dam",
    )
)

fig.update_layout(
    title="Band 1 (Blue, 0.47 μm) radiance — fixed-grid coordinates",
    width=900,
    height=500,
)

path = project_root() / "figures"
fig.write_html(path / "radiance_with_hoover_dam.html")

In [19]:
# Build the GOES projection for cartopy
proj_info = ds["goes_imager_projection"]
goes_crs = ccrs.Geostationary(
    central_longitude=proj_info.attrs["longitude_of_projection_origin"],
    satellite_height=proj_info.attrs["perspective_point_height"],
    sweep_axis=proj_info.attrs["sweep_angle_axis"],
)

In [20]:
# Convert fixed-grid x,y (radians) to meters for cartopy
h = proj_info.attrs["perspective_point_height"]
x_m = ds["x"].values * h
y_m = ds["y"].values * h

In [25]:
# Radiance data
rad = ds["Rad"].values

# Build figure
fig = go.Figure()

# 1. Add radiance image (GOES projected grid)
fig.add_trace(
    go.Heatmap(
        z=rad,
        x=x_m,
        y=y_m,
        colorscale="Viridis",
        colorbar=dict(title="Radiance"),
        showscale=True,
    )
)

# 2. Add W A Parish reference point (lon/lat)
fig.add_trace(
    go.Scattergeo(
        lon=[-114.7372],
        lat=[36.0162],
        mode="markers",
        marker=dict(size=12, color="red", symbol="x", line=dict(width=3)),
        name="Hoover Dam (NV)",
    )
)

# 3. Configure geographic layers (coastlines, borders, states)
fig.update_geos(
    resolution=50,
    showcountries=True,
    countrycolor="white",
    showsubunits=True,
    subunitcolor="white",
    showcoastlines=True,
    coastlinecolor="white",
    projection_type="mercator",
)

# 4. Layout
fig.update_layout(
    title="Band 1 radiance with coastlines and Hoover dam location",
    width=1000,
    height=700,
    geo=dict(
        scope="north america",
        projection_scale=3,
        center=dict(lat=36.0162, lon=-114.7372),
        showland=True,
        landcolor="black",
    ),
)

path = project_root() / "figures"
fig.write_html(path / "radiance_with_hoover_dam_on_the_map.html")